# Import and Setup

In [ ]:
import os
import json
import ast
from openai import OpenAI

openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key
client = OpenAI()
MODEL = "gpt-5.4"
REASONING_EFFORT = "medium"  # "low", "medium", "high", or None to disable

# LLM Functions

In [2]:
REQUIREMENTS = """- Must run quickly on a MacBook with 36GB RAM (Apple Silicon); use device='mps' where supported
- Use a single small transformer-based model (e.g. distilbert, all-MiniLM-L6-v2, or similarly lightweight models via transformers.pipeline or sentence-transformers)
- No training, fine-tuning, or weight updates — load a pretrained model and evaluate it directly (zero-shot or task-specific pretrained checkpoint)
- No hyperparameter tuning or loops over multiple models/configurations — pick one and run it
- Only one model and one dataset/subset
- Only code cells (no markdown cells)
- No plots or visualizations"""


In [3]:
def generate_data_science_tasks(n: int = 10) -> list:
    prompt = f"""Brainstorm a list of {n} descriptions of AI tasks that can be evaluated using a modern AI model and HuggingFace datasets.

Requirements for each task:
- Restrict to tasks with datasets that have less than a million samples
- Each description should specify both the task type and the dataset

Return ONLY a valid Python list of strings, no explanation."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = response.choices[0].message.content.strip()
    return ast.literal_eval(raw)


tasks = generate_data_science_tasks(10)
tasks

['Sentiment analysis on the IMDb dataset (binary movie review classification, 50k samples).',
 'Natural language inference on the SNLI dataset (entailment, contradiction, neutral classification, about 570k samples).',
 'Question answering on the SQuAD v1.1 dataset (extractive reading comprehension, about 100k question-answer pairs).',
 'Paraphrase identification on the MRPC dataset from GLUE (sentence pair binary classification, about 5.8k samples).',
 'Topic classification on the AG News dataset (4-class news article classification, 120k training samples).',
 'Named entity recognition on the CoNLL-2003 dataset (token classification for PER/ORG/LOC/MISC entities, about 23k sentences).',
 'Summarization on the XSum dataset (single-document abstractive summarization, about 226k samples).',
 'Machine translation on the IWSLT 2017 German-English dataset (sequence-to-sequence translation, well under 1 million sentence pairs).',
 'Toxic comment classification on the civil_comments dataset (m

In [ ]:
def _parse_notebook_raw(raw: str) -> dict:
    """Strip markdown fences and parse notebook JSON."""
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("```", 2)[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.rsplit("```", 1)[0].strip()
    return json.loads(raw)


def _call(messages: list, **kwargs) -> str:
    """Make a model call, optionally with reasoning_effort."""
    extra = {"reasoning_effort": REASONING_EFFORT} if REASONING_EFFORT is not None else {}
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        **extra,
        **kwargs,
    )
    return response.choices[0].message.content


def generate_methodology_suite(task: str, notebook_dir: str, n: int = 5) -> list:
    """
    Plans n methodologically distinct notebooks for a task and generates each one.
    All notebooks are treated as peers — there is no 'original' and no 'variations'.
    Methodology diversity is enforced upfront in the planning step.
    """
    os.makedirs(notebook_dir, exist_ok=True)

    # Step 1: Plan n distinct methodology approaches
    print(f"=== Step 1: Planning {n} methodologically distinct notebooks ===")
    raw_plans = _call([{"role": "user", "content": f"""You are an expert ML engineer. Plan {n} Jupyter notebooks that each implement a different end-to-end ML pipeline for the following task.

Task: {task}

Your goal is to explore the space of correct machine learning pipeline methodologies for this task. 


Requirements that every notebook must satisfy:
{REQUIREMENTS}

For each notebook, write:
- A snake_case filename (without .ipynb)
- A methodology summary: one sentence stating the inference entry point, representation strategy, and scoring approach
- A step-by-step implementation plan

Return ONLY a valid Python dictionary in exactly this format, no explanation outside the dict:
{{
  "notebook_name": {{
    "methodology": "one-sentence methodology summary",
    "plan": "step-by-step plan as a single string"
  }},
  ...
}}"""}])
    print(f"Plans:\n{raw_plans}\n")
    suite_plans = ast.literal_eval(raw_plans)

    # Step 2: Generate each notebook
    paths = []
    for name, spec in suite_plans.items():
        methodology = spec["methodology"]
        plan = spec["plan"]
        print(f"--- Generating: '{name}' ---")
        print(f"Methodology: {methodology}\n")

        raw = _call([{"role": "user", "content": f"""You are an expert ML engineer. Generate a complete Jupyter notebook as valid JSON for the following task and pipeline methodology.

Task: {task}

Pipeline methodology: {methodology}

Implementation plan:
{plan}

Requirements:
{REQUIREMENTS}
- Use HuggingFace datasets to load data
- Include cells for imports, data loading, inference, and evaluation
- Implement exactly the pipeline methodology described above — do not substitute a simpler or different approach
- The notebook must be valid .ipynb JSON (nbformat 4)
- Return ONLY the raw JSON, no markdown fences or explanation."""}])
        print(f"Raw JSON (first 300 chars): {raw[:300]}\n")

        nb = _parse_notebook_raw(raw)
        path = os.path.join(notebook_dir, f"{name}.ipynb")
        with open(path, "w") as f:
            json.dump(nb, f, indent=1)
        print(f"Saved: {path}")
        paths.append(path)

    return paths

# Generate Batch

In [5]:
selected_task = "Semantic textual similarity scoring on the STS-B"  # replace with desired task
print(selected_task)

Paraphrase identification on the MRPC dataset from GLUE


In [ ]:
import shutil

NOTEBOOK_DIR = "../notebooks/batch_3"
if os.path.exists(NOTEBOOK_DIR):
    shutil.rmtree(NOTEBOOK_DIR)
os.makedirs(NOTEBOOK_DIR)

paths = generate_methodology_suite(selected_task, NOTEBOOK_DIR, n=5)
print(f"\nGenerated {len(paths)} notebooks.")